In [ ]:
%load_ext autoreload
%autoreload 2

import anndata as ad
adata = ad.read_h5ad("./data/larry/larry_processed.h5ad")
adata_cospar = ad.read_h5ad("./data/larry/cospar_tmap_result.h5ad")
print(adata)
print(adata_cospar)

In [ ]:
import numpy as np

def match_by_X_emb(adata, adata_cospar):
    X1 = adata.obsm["X_emb"]
    X2 = adata_cospar.obsm["X_emb"]

    # Convert rows to tuples (hashable, exact float match)
    rows1 = [tuple(r) for r in X1]
    rows2 = [tuple(r) for r in X2]

    # Build lookup: embedding -> index in adata_cospar
    lookup = {row: j for j, row in enumerate(rows2)}

    # Map each row in adata → adata_cospar index (or -1 if missing)
    mapA2B = np.array([lookup.get(row, -1) for row in rows1], dtype=int)

    return mapA2B

mapA2B = match_by_X_emb(adata, adata_cospar)
print("Matched:", np.sum(mapA2B != -1), "/", len(mapA2B))

In [ ]:
from scipy.spatial import cKDTree
import joblib

# ===============================================================
# 0. Load embedder
# ===============================================================
emb = joblib.load("./data/larry/larry_embedder.pkl")
emb.gene_names = adata.var_names
emb.cospar_index = mapA2B     # your mapping (adata → adata_cospar)


# ===============================================================
# 1. Tentatively assign new_X_emb from FlowMap
# ===============================================================
n_cospar = adata_cospar.n_obs
new_X_emb = np.full((n_cospar, 2), np.nan)   # initialize with NaNs

idx = emb.cospar_index
mask = idx != -1                             # valid matches only

# fill matched rows
new_X_emb[idx[mask]] = emb.X_emb[mask]


# ===============================================================
# 2. Identify rows with NaN in new_X_emb
# ===============================================================
nan_rows = np.where(~np.isfinite(new_X_emb).all(axis=1))[0]
print("Cells needing patching:", len(nan_rows))


# ===============================================================
# 3. Find nearest neighbors in *original CoSpar embedding*
# ===============================================================
X_cospar_orig = adata_cospar.obsm["X_emb"]    # THIS is the reference geometry

# Use only rows whose new_X_emb is NOT NaN
valid_rows = np.where(np.isfinite(new_X_emb).all(axis=1))[0]

# KD-tree in original embedding space, but only valid rows
tree = cKDTree(X_cospar_orig[valid_rows])

# query NN for each NaN row
_, nn_idx = tree.query(X_cospar_orig[nan_rows], k=1)

# convert relative NN indices → global indices
global_nn_idx = valid_rows[nn_idx]


# ===============================================================
# 4. Replace NaN rows in new_X_emb with the NN's embedding
# ===============================================================
new_X_emb[nan_rows] = new_X_emb[global_nn_idx]


# ===============================================================
# 5. Store the patched embedding back into CoSpar object
# ===============================================================
adata_cospar.obsm["X_emb"] = new_X_emb

print("DONE: adata_cospar.X_emb refreshed and patched.")

In [ ]:
import matplotlib.pyplot as plt
from scripts.FlowCurvature import compute_flow_curvature
from scripts.plotting import *

# ==============================================================
# 0. Compute curvature + acceleration decomposition
# ==============================================================
X_emb = adata_cospar.obsm["X_emb"]
curv = compute_flow_curvature(emb, X_emb)

A        = curv["A"]          # (N, D)
A_tan    = curv["A_tan"]      # (N, d)
A_along  = curv["A_along"]    # (N, D)
A_steer  = curv["A_steer"]    # (N, d)
A_nor    = curv["A_nor"]      # (N, D)

# Raw magnitudes
A_norm        = np.linalg.norm(A, axis=1)
A_tan_norm    = np.linalg.norm(A_tan, axis=1)
A_along_norm    = np.linalg.norm(A_along, axis=1)
A_steer_norm  = np.linalg.norm(A_steer, axis=1)
A_nor_norm    = np.linalg.norm(A_nor,   axis=1)

# ==============================================================
# 1. Quantile clipping helper
# ==============================================================

def clip_quantile(arr, q_low=2, q_high=98):
    lo, hi = np.percentile(arr, [q_low, q_high])
    return np.clip(arr, lo, hi)

speed  = np.linalg.norm(curv["V"], axis=1) + 1e-12
speed2 = speed**2

# curvature-scaled magnitudes
A_curv_norm       = A_norm       / speed2
A_tan_curv_norm   = A_tan_norm   / speed2
A_along_norm      = A_along_norm / speed2
A_steer_curv_norm = A_steer_norm / speed2
A_nor_curv_norm   = A_nor_norm   / speed2

# clipped for visualization
A_curv_c       = clip_quantile(A_curv_norm)
A_tan_curv_c   = clip_quantile(A_tan_curv_norm)
A_along_curv_c   = clip_quantile(A_along_norm)
A_steer_curv_c = clip_quantile(A_steer_curv_norm)
A_nor_curv_c   = clip_quantile(A_nor_curv_norm)

# ==============================================================
# 2. Plot curvature-scaled components
# ==============================================================

common_kwargs = dict(
    tps_vf=emb.tps_vf,
    stream_density=0.9,
    streamline_thickness=5.0,
    arrowsize=1.5,
    scatter_size=8,
    scatter_alpha=0.6,
    cmap="coolwarm",
    show_axes=False
)

# --- total acceleration curvature ---
plot_velocity_streamplot(
    X_emb,
    scatter_color=A_curv_c,
    **common_kwargs
)

# --- tangent acceleration curvature ---
plot_velocity_streamplot(
    X_emb,
    scatter_color=A_tan_curv_c,
    **common_kwargs
)

plot_velocity_streamplot(
    X_emb,
    scatter_color=A_along_curv_c,
    **common_kwargs
)

# --- steering (geodesic) curvature contribution ---
plot_velocity_streamplot(
    X_emb,
    scatter_color=A_steer_curv_c,
    **common_kwargs
)

# --- normal (extrinsic) curvature contribution ---
plot_velocity_streamplot(
    X_emb,
    scatter_color=A_nor_curv_c,
    **common_kwargs
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors

from scripts.plotting import compute_velocity_on_grid


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------
def points_inside_mask(X, seeds, k=8, radius_scale=1.2):
    nn = NearestNeighbors(n_neighbors=k).fit(X)
    r = np.median(nn.kneighbors(X)[0][:, -1]) * radius_scale
    neigh = nn.radius_neighbors(seeds, radius=r, return_distance=False)
    return np.array([len(ix) > 0 for ix in neigh])


# -----------------------------------------------------------------------------
# Data
# -----------------------------------------------------------------------------
curvature = A_steer_curv_c
V_cells = emb.tps_vf.predict(X_emb)


# -----------------------------------------------------------------------------
# Velocity grid (sparse but visible)
# -----------------------------------------------------------------------------
Xg, _, _ = compute_velocity_on_grid(
    X_emb,
    grid_size=25,
    min_mass=0.02
)

keep = points_inside_mask(X_emb, Xg)
Xg = Xg[keep][::2]
Vg = emb.tps_vf.predict(Xg)


# -----------------------------------------------------------------------------
# Plot
# -----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 6))

# Cells (main signal)
ax.scatter(
    X_emb[:, 0], X_emb[:, 1],
    c=curvature,
    cmap="coolwarm",
    s=20,              # ↓ smaller dots
    alpha=0.15,
    linewidths=0,
    zorder=1
)

# Velocity field (slightly more visible)
ax.quiver(
    Xg[:, 0], Xg[:, 1],
    Vg[:, 0], Vg[:, 1],
    angles="xy",
    scale_units="xy",
    scale=2.5,        # ↓ smaller scale → longer arrows (was 4.0)
    width=0.006,      # ↑ slightly thicker (was 0.0045)
    color="k",
    alpha=0.8,        # slightly darker
    headwidth=5.0,    # slightly bigger heads
    headlength=4.5,
    headaxislength=3.8,
    zorder=2
)


# -----------------------------------------------------------------------------
# Formatting
# -----------------------------------------------------------------------------
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()

plt.savefig(
    "./figures/larry/larry_quiver.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# -----------------------------------------------------------------------------
# Compute Jacobians at cell locations
# -----------------------------------------------------------------------------
J = emb.tps.compute_jacobians(X_emb)   # (N, d_high, 2)

# -----------------------------------------------------------------------------
# Project first
# -----------------------------------------------------------------------------
A_emb_raw = np.zeros((X_emb.shape[0], 2))

for i in range(X_emb.shape[0]):
    A_emb_raw[i], _, _, _ = np.linalg.lstsq(J[i], A_tan[i], rcond=None)

# -----------------------------------------------------------------------------
# Normalize in embedding space
# -----------------------------------------------------------------------------
A_emb = A_emb_raw / (speed2[:, None] + 1e-12)

# -----------------------------------------------------------------------------
# Subsample for TPS fitting
# -----------------------------------------------------------------------------
n_fit = 4000

idx = np.random.choice(X_emb.shape[0], size=n_fit, replace=False)

X_sub = X_emb[idx]
A_sub = A_emb[idx]

# -----------------------------------------------------------------------------
# Fit TPS on subset
# -----------------------------------------------------------------------------
from scripts.TPS import ThinPlateSpline

tps_A = ThinPlateSpline(X_sub, **emb.tps_vf_kwargs)
tps_A.fit(A_sub, dof=emb.dof_vf)

# -----------------------------------------------------------------------------
# Grid
# -----------------------------------------------------------------------------
Xg, _, _ = compute_velocity_on_grid(
    X_emb,
    grid_size=25,
    min_mass=0.02
)

keep = points_inside_mask(X_emb, Xg)
Xg = Xg[keep][::2]

# -----------------------------------------------------------------------------
# Evaluate acceleration field
# -----------------------------------------------------------------------------
Ag = tps_A.predict(Xg)

fig, ax = plt.subplots(figsize=(6, 6))

# Cells
ax.scatter(
    X_emb[:, 0], X_emb[:, 1],
    c=A_tan_curv_c,
    cmap="coolwarm",
    s=20,
    alpha=0.15,
    linewidths=0,
    zorder=1
)

# Acceleration field
ax.quiver(
    Xg[:, 0], Xg[:, 1],
    Ag[:, 0], Ag[:, 1],
    angles="xy",
    scale_units="xy",
    scale=0.04,
    width=0.006,
    color="k",
    alpha=0.8,
    headwidth=2.5,        # ↓ smaller (was 5.0)
    headlength=2.5,       # ↓ smaller (was 4.5)
    headaxislength=2.,   # ↓ smaller (was 3.8)
    zorder=2
)

# Formatting
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])

for spine in ax.spines.values():
    spine.set_visible(False)

# Add margin
xmin, xmax = X_emb[:, 0].min(), X_emb[:, 0].max()
xpad = 0.1 * (xmax - xmin)
ax.set_xlim(xmin - xpad, xmax + xpad)

plt.tight_layout()
plt.savefig("./figures/larry/acceleration_quiver.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# -----------------------------------------------------------------------------
# Compute Jacobians at cell locations
# -----------------------------------------------------------------------------
J = emb.tps.compute_jacobians(X_emb)   # (N, d_high, 2)

# -----------------------------------------------------------------------------
# Project first
# -----------------------------------------------------------------------------
A_emb_raw = np.zeros((X_emb.shape[0], 2))

for i in range(X_emb.shape[0]):
    A_emb_raw[i], _, _, _ = np.linalg.lstsq(J[i], A_along[i], rcond=None)

# -----------------------------------------------------------------------------
# Normalize in embedding space
# -----------------------------------------------------------------------------
A_emb = A_emb_raw / (speed2[:, None] + 1e-12)

# -----------------------------------------------------------------------------
# Subsample for TPS fitting
# -----------------------------------------------------------------------------
n_fit = 4000

idx = np.random.choice(X_emb.shape[0], size=n_fit, replace=False)

X_sub = X_emb[idx]
A_sub = A_emb[idx]

# -----------------------------------------------------------------------------
# Fit TPS on subset
# -----------------------------------------------------------------------------
from scripts.TPS import ThinPlateSpline

tps_A = ThinPlateSpline(X_sub, **emb.tps_vf_kwargs)
tps_A.fit(A_sub, dof=emb.dof_vf)

# -----------------------------------------------------------------------------
# Grid
# -----------------------------------------------------------------------------
Xg, _, _ = compute_velocity_on_grid(
    X_emb,
    grid_size=25,
    min_mass=0.02
)

keep = points_inside_mask(X_emb, Xg)
Xg = Xg[keep][::2]

# -----------------------------------------------------------------------------
# Evaluate acceleration field
# -----------------------------------------------------------------------------
Ag = tps_A.predict(Xg)

fig, ax = plt.subplots(figsize=(6, 6))

# Cells
ax.scatter(
    X_emb[:, 0], X_emb[:, 1],
    c=A_along_curv_c,
    cmap="coolwarm",
    s=20,
    alpha=0.15,
    linewidths=0,
    zorder=1
)

# Acceleration field
ax.quiver(
    Xg[:, 0], Xg[:, 1],
    Ag[:, 0], Ag[:, 1],
    angles="xy",
    scale_units="xy",
    scale=0.04,
    width=0.006,
    color="k",
    alpha=0.8,
    headwidth=2.5,        # ↓ smaller (was 5.0)
    headlength=2.5,       # ↓ smaller (was 4.5)
    headaxislength=2.,   # ↓ smaller (was 3.8)
    zorder=2
)

# Formatting
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])

for spine in ax.spines.values():
    spine.set_visible(False)

# Add margin
xmin, xmax = X_emb[:, 0].min(), X_emb[:, 0].max()
xpad = 0.1 * (xmax - xmin)
ax.set_xlim(xmin - xpad, xmax + xpad)

plt.tight_layout()
plt.savefig("./figures/larry/acceleration_quiver.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# -----------------------------------------------------------------------------
# Compute Jacobians at cell locations
# -----------------------------------------------------------------------------
J = emb.tps.compute_jacobians(X_emb)   # (N, d_high, 2)

# -----------------------------------------------------------------------------
# Project first
# -----------------------------------------------------------------------------
A_emb_raw = np.zeros((X_emb.shape[0], 2))

for i in range(X_emb.shape[0]):
    A_emb_raw[i], _, _, _ = np.linalg.lstsq(J[i], A_steer[i], rcond=None)

# -----------------------------------------------------------------------------
# Normalize in embedding space
# -----------------------------------------------------------------------------
A_emb = A_emb_raw / (speed2[:, None] + 1e-12)

# -----------------------------------------------------------------------------
# Subsample for TPS fitting
# -----------------------------------------------------------------------------
n_fit = 4000

idx = np.random.choice(X_emb.shape[0], size=n_fit, replace=False)

X_sub = X_emb[idx]
A_sub = A_emb[idx]

# -----------------------------------------------------------------------------
# Fit TPS on subset
# -----------------------------------------------------------------------------
from scripts.TPS import ThinPlateSpline

tps_A = ThinPlateSpline(X_sub, **emb.tps_vf_kwargs)
tps_A.fit(A_sub, dof=emb.dof_vf)

# -----------------------------------------------------------------------------
# Grid
# -----------------------------------------------------------------------------
Xg, _, _ = compute_velocity_on_grid(
    X_emb,
    grid_size=25,
    min_mass=0.02
)

keep = points_inside_mask(X_emb, Xg)
Xg = Xg[keep][::2]

# -----------------------------------------------------------------------------
# Evaluate acceleration field
# -----------------------------------------------------------------------------
Ag = tps_A.predict(Xg)

fig, ax = plt.subplots(figsize=(6, 6))

# Cells
ax.scatter(
    X_emb[:, 0], X_emb[:, 1],
    c=A_steer_curv_c,
    cmap="coolwarm",
    s=20,
    alpha=0.15,
    linewidths=0,
    zorder=1
)

# Acceleration field
ax.quiver(
    Xg[:, 0], Xg[:, 1],
    Ag[:, 0], Ag[:, 1],
    angles="xy",
    scale_units="xy",
    scale=0.04,
    width=0.006,
    color="k",
    alpha=0.8,
    headwidth=2.5,        # ↓ smaller (was 5.0)
    headlength=2.5,       # ↓ smaller (was 4.5)
    headaxislength=2.,   # ↓ smaller (was 3.8)
    zorder=2
)

# Formatting
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])

for spine in ax.spines.values():
    spine.set_visible(False)

# Add margin
xmin, xmax = X_emb[:, 0].min(), X_emb[:, 0].max()
xpad = 0.1 * (xmax - xmin)
ax.set_xlim(xmin - xpad, xmax + xpad)

plt.tight_layout()
plt.savefig("./figures/larry/acceleration_quiver.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

def grow_clusters_with_boundary(
    coords,
    vals,
    seed_points,
    boundary_fn,
    k=5,
    m=0.12
):
    """
    Region-growing clustering with a hard geometric boundary.

    boundary_fn(pt) -> signed value
        same sign = same allowed region
    """

    # --------------------------------------------------
    # 1) find seed indices + seed sides
    # --------------------------------------------------
    seed_idx = []
    seed_side = []

    for p in seed_points:
        dists = np.linalg.norm(coords - p, axis=1)
        idx = np.argmin(dists)
        seed_idx.append(idx)
        seed_side.append(np.sign(boundary_fn(p)))

    seed_idx  = np.array(seed_idx)
    seed_side = np.array(seed_side)

    # --------------------------------------------------
    # 2) KNN graph
    # --------------------------------------------------
    nn = NearestNeighbors(n_neighbors=k + 1).fit(coords)
    nbrs = nn.kneighbors(return_distance=False)[:, 1:]

    # --------------------------------------------------
    # 3) Region growing
    # --------------------------------------------------
    N = coords.shape[0]
    labels = -np.ones(N, dtype=int)
    stack = []

    for cid, s in enumerate(seed_idx):
        labels[s] = cid
        stack.append(s)

    while stack:
        i = stack.pop()
        cid = labels[i]
        side = seed_side[cid]

        for j in nbrs[i]:

            if labels[j] != -1:
                continue

            if vals[j] <= m:
                continue

            if np.sign(boundary_fn(coords[j])) != side:
                continue

            labels[j] = cid
            stack.append(j)

    return seed_idx, labels

In [ ]:
coords = X_emb
vals   = A_steer_curv_c

# zoom region
x_min, x_max = 1.0, 4.5
y_min, y_max = 2.5, 5.5

mask_zoom = (
    (coords[:, 0] > x_min) & (coords[:, 0] < x_max) &
    (coords[:, 1] > y_min) & (coords[:, 1] < y_max)
)

coords_zoom = coords[mask_zoom]
vals_zoom   = vals[mask_zoom]

# hard boundary: x + y = 6.6
def boundary_fn(pt):
    return (pt[0] + pt[1]) - 6.6

seed_points = np.array([
    [2.0, 3.5],   # below boundary
    [3.3, 4.5],   # above boundary
])

seed_idx, labels_grow = grow_clusters_with_boundary(
    coords_zoom,
    vals_zoom,
    seed_points,
    boundary_fn,
    k=5,
    m=0.12
)

fig, ax = plt.subplots(figsize=(6, 6))

mask_assigned = labels_grow != -1

# unassigned
ax.scatter(
    coords_zoom[~mask_assigned, 0],
    coords_zoom[~mask_assigned, 1],
    c="lightgray",
    s=15,
    alpha=0.3,
    linewidths=0,
    zorder=1
)

# assigned clusters
ax.scatter(
    coords_zoom[mask_assigned, 0],
    coords_zoom[mask_assigned, 1],
    c=labels_grow[mask_assigned],
    cmap="tab10",
    s=30,
    alpha=0.85,
    linewidths=0,
    zorder=2
)

ax.set_aspect("equal")
# ax.set_xticks([]); ax.set_yticks([])
# for spine in ax.spines.values():
#     spine.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

sc = ax.scatter(
    coords_zoom[:, 0],
    coords_zoom[:, 1],
    c=vals_zoom,
    cmap="coolwarm",
    s=160,
    alpha=0.45,
    linewidths=0,
    zorder=1
)

ax.set_aspect("equal")
# ax.set_xticks([]); ax.set_yticks([])
# for spine in ax.spines.values():
#     spine.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
from scripts.least_action_path import *

emb.spline = emb.tps
emb.spline_vf = emb.tps_vf
lap = LagrangianPathOptimizer(
    emb,
    D=1.0,
    lam=0.0,
)

path_init = np.array([
    [3.6, 2.8],
    [2.5, 4.2],
    [1.7, 4.0],
    [1.0, 4.2],
])

lap_result = lap.fit_path(
    path_init=path_init,
    n_segments=100,
    lr=5e-3,
    iters=300,
)

lap_curve = lap_result["path_refined"]
lap_init = lap_result["path_init"]
dt = lap_result["dt"]

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

sc = ax.scatter(
    coords_zoom[:, 0],
    coords_zoom[:, 1],
    c=vals_zoom,
    cmap="coolwarm",
    s=160,
    alpha=0.45,
    linewidths=0,
    zorder=1
)

# --------------------------------------------------
# Overlay LAP curve
# --------------------------------------------------
ax.plot(
    lap_init[:, 0], lap_init[:, 1],
    "--", color="black", lw=2,
    alpha=0.7,
    zorder=3,
    label="init"
)

ax.plot(
    lap_curve[:, 0], lap_curve[:, 1],
    "-", color="limegreen", lw=3,
    zorder=4,
    label="LAP"
)

# mark anchor points
ax.scatter(
    path_init[:, 0], path_init[:, 1],
    s=120, c="red",
    edgecolors="black",
    zorder=5
)

ax.set_aspect("equal")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

curve = lap_curve.copy()
eps = 2

Xq = X_emb.copy()
Vq = curv["V"].copy()
Asteer = A_steer.copy()
Aalong = A_along.copy()

seg0 = curve[:-1]
seg1 = curve[1:]
segv = seg1 - seg0
seglen2 = np.sum(segv**2, axis=1) + 1e-12
seglen = np.sqrt(seglen2)

tree = cKDTree(curve)
d_vertex, idx_vertex = tree.query(Xq, k=1)

candidate_segs = np.unique(
    np.clip(
        np.r_[
            idx_vertex - 2,
            idx_vertex - 1,
            idx_vertex,
            idx_vertex + 1,
            idx_vertex + 2,
        ],
        0,
        len(seg0) - 1,
    )
)

best_dist2 = np.full(Xq.shape[0], np.inf)
best_seg = np.full(Xq.shape[0], -1, dtype=int)
best_t = np.zeros(Xq.shape[0])
best_proj = np.zeros_like(Xq)

for s in candidate_segs:
    p0 = seg0[s]
    v = segv[s]
    t = np.sum((Xq - p0) * v, axis=1) / seglen2[s]
    t = np.clip(t, 0.0, 1.0)
    proj = p0 + t[:, None] * v
    dist2 = np.sum((Xq - proj) ** 2, axis=1)
    keep = dist2 < best_dist2
    best_dist2[keep] = dist2[keep]
    best_seg[keep] = s
    best_t[keep] = t[keep]
    best_proj[keep] = proj[keep]

dist_curve = np.sqrt(best_dist2)
near_mask = dist_curve <= eps
near_idx = np.where(near_mask)[0]

curve_arclen = np.r_[0.0, np.cumsum(seglen)]
curve_s = curve_arclen[best_seg] + best_t * seglen[best_seg]
curve_s_norm = curve_s / (curve_arclen[-1] + 1e-12)

speed = np.linalg.norm(Vq, axis=1) + 1e-12
speed2 = speed**2

steer_mag_raw = np.linalg.norm(Asteer, axis=1)
steer_curv_mag = steer_mag_raw / speed2

J = emb.tps.compute_jacobians(X_emb)

V_emb = np.zeros((X_emb.shape[0], 2))
Aalong_emb = np.zeros((X_emb.shape[0], 2))
Asteer_emb = np.zeros((X_emb.shape[0], 2))

for i in range(X_emb.shape[0]):
    V_emb[i], _, _, _ = np.linalg.lstsq(J[i], curv["V"][i], rcond=None)
    Aalong_emb[i], _, _, _ = np.linalg.lstsq(J[i], A_along[i], rcond=None)
    Asteer_emb[i], _, _, _ = np.linalg.lstsq(J[i], A_steer[i], rcond=None)

speed = np.linalg.norm(V_emb, axis=1) + 1e-12
speed2 = speed**2

steer_mag_raw = np.linalg.norm(Asteer_emb, axis=1)
steer_curv_mag = steer_mag_raw / speed2

cross_z = Aalong_emb[:, 0] * Asteer_emb[:, 1] - Aalong_emb[:, 1] * Asteer_emb[:, 0]
steer_orientation = np.sign(cross_z)

curve_df = pd.DataFrame({
    "cell_idx": near_idx,
    "x": Xq[near_idx, 0],
    "y": Xq[near_idx, 1],
    "proj_x": best_proj[near_idx, 0],
    "proj_y": best_proj[near_idx, 1],
    "dist_curve": dist_curve[near_idx],
    "curve_seg": best_seg[near_idx],
    "curve_t": best_t[near_idx],
    "curve_s": curve_s[near_idx],
    "curve_s_norm": curve_s_norm[near_idx],
    "speed": speed[near_idx],
    "steer_mag_raw": steer_mag_raw[near_idx],
    "steer_curv_mag": steer_curv_mag[near_idx],
    "steer_orientation": steer_orientation[near_idx],
})

dx =  best_proj[:, 0] - X_emb[:, 0]
dy =  best_proj[:, 1] - X_emb[:, 1]

dist_signed = np.sqrt(dx**2 + dy**2) * np.sign(
    dx * (segv[best_seg][:, 1]) - dy * (segv[best_seg][:, 0])
)

curve_df["dist_signed"] = dist_signed[near_idx]
curve_df["ax"] = steer_curv_mag[near_idx] * steer_orientation[near_idx]
curve_df["ay"] = np.zeros_like(curve_df["ax"])

curve_df.head()

In [ ]:
import matplotlib.pyplot as plt

x = curve_df["dist_signed"].values
y = curve_df["curve_s"].values
c = curve_df["steer_curv_mag"].values

# optional clipping / filtering
lo_y, hi_y = np.percentile(y, [1, 99])
mask = (y > lo_y) & (y < hi_y)

x = x[mask]
y = y[mask]
c = c[mask]

lo_c, hi_c = np.percentile(c, [2, 98])
c = np.clip(c, lo_c, hi_c)

u = curve_df["ax"].values[mask]
v = curve_df["ay"].values[mask]

fig, ax = plt.subplots(figsize=(6, 8))

# --------------------------------------------------
# Scatter (background field)
# --------------------------------------------------
sc = ax.scatter(
    x,
    y,
    c=c,
    cmap="coolwarm",
    s=100,
    alpha=0.25,
    linewidths=0,
    edgecolors="none",
    zorder=1
)

# --------------------------------------------------
# Subsample for quiver (KEY STEP)
# --------------------------------------------------
step = 10   # try 3–6 depending on density
idx = np.arange(len(x))[::step]

x_q = x[idx]
y_q = y[idx]
u_q = u[idx]
v_q = v[idx]

# --------------------------------------------------
# Acceleration vectors (clean + readable)
# --------------------------------------------------
ax.quiver(
    x_q,
    y_q,
    u_q,
    v_q,
    angles="xy",
    scale_units="xy",

    scale=1.5,        # ↓ smaller = longer arrows (this is the main one)
    width=0.008,      # ↑ thicker shaft
    headwidth=6,      # ↑ wider head
    headlength=7,     # ↑ longer head
    headaxislength=6, # cleaner shape

    color="#2c3e50",
    alpha=0.85,
    zorder=3
)

# --------------------------------------------------
# Bounds + padding
# --------------------------------------------------
ymin, ymax = y.min(), y.max()
pad = 0.03 * (ymax - ymin)

# --------------------------------------------------
# Reference line (LAP axis → purple)
# --------------------------------------------------
ax.vlines(
    0,
    ymin + pad,
    ymax - pad,
    color="#6a3d9a",
    lw=2.5,
    zorder=2
)

# --------------------------------------------------
# Top / Bottom bounds (slightly offset)
# --------------------------------------------------
ax.hlines(
    [ymin - pad*0.5, ymax + pad*0.5],
    xmin=x.min(),
    xmax=x.max(),
    color="black",
    lw=1.0,
    alpha=0.6,
    zorder=0
)

# --------------------------------------------------
# Clean styling
# --------------------------------------------------
ax.set_xticks([])
ax.set_yticks([])

for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_aspect("auto")

plt.tight_layout()
plt.savefig("./figures/larry/lap_path_1_acceleration.pdf")
plt.show()

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

# subset COSPAR AnnData
adata_cospar_zoom = adata_cospar[mask_zoom].copy()
adata_cospar_zoom.obs["cluster"] = labels_grow.astype(str)

adata_cospar_zoom.raw = adata_cospar_zoom
adata_cospar_zoom.obsm["X_umap"] = adata_cospar_zoom.obsm["X_emb"]

# differential expression
cluster_A, cluster_B = "0", "1"

sc.tl.rank_genes_groups(
    adata_cospar_zoom,
    groupby="cluster",
    groups=[cluster_A],
    reference=cluster_B,
    method="wilcoxon",
)

print(adata_cospar_zoom.obs["cluster"].value_counts(), "\n")

# collect DE results
de = adata_cospar_zoom.uns["rank_genes_groups"]

deg_df = (
    pd.DataFrame({
        "gene": de["names"][cluster_A],
        "score": de["scores"][cluster_A],
        "logFC": de["logfoldchanges"][cluster_A],
        "pval": de["pvals"][cluster_A],
        "pval_adj": de["pvals_adj"][cluster_A],
    })
    .dropna()
    .sort_values("pval_adj")
)

print(deg_df.head(20))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ---------------------------------------
# 0. Select genes
# ---------------------------------------
genes = (
    deg_df
    .sort_values("pval_adj")
    .head(12)["gene"]   # start with 12 for clean grid
    .tolist()
)

genes = [g for g in genes if g in adata_cospar.var_names]

# ---------------------------------------
# 1. Embedding
# ---------------------------------------
X_emb = adata_cospar.obsm["X_emb"]
x = X_emb[:, 0]
y = X_emb[:, 1]

# ---------------------------------------
# 2. Expression
# ---------------------------------------
X = adata_cospar[:, genes].X
if hasattr(X, "A"):
    X = X.A

# ---------------------------------------
# 3. Plot grid
# ---------------------------------------
n = len(genes)
ncols = 4
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3*nrows))

axes = axes.flatten()

for i, g in enumerate(genes):
    ax = axes[i]
    
    vals = X[:, i]
    
    # optional clipping for better contrast
    lo, hi = np.percentile(vals, [2, 98])
    vals = np.clip(vals, lo, hi)
    
    sc = ax.scatter(
        x,
        y,
        c=vals,
        cmap="viridis",
        s=8,
        alpha=0.8,
        linewidths=0,
        rasterized=True
    )
    
    ax.set_title(g, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    
    for spine in ax.spines.values():
        spine.set_visible(False)

# remove empty panels
for j in range(i+1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ---------------------------------------------------
# 0. Select genes
# ---------------------------------------------------
genes = (
    deg_df
    .sort_values("pval_adj")
    .head(25)["gene"]
    .tolist()
)

genes = [g for g in genes if g in adata_cospar.var_names]

# ---------------------------------------------------
# 1. Extract expression
# ---------------------------------------------------
X = adata_cospar[:, genes].X
if hasattr(X, "A"):
    X = X.A

df = pd.DataFrame(X, columns=genes)

# ---------------------------------------------------
# 2. Cluster-level averages (A vs B)
# ---------------------------------------------------
cluster_labels = adata_cospar_zoom.obs["cluster"].astype(str)

mask_A = (cluster_labels == "0")
mask_B = (cluster_labels == "1")

cluster_mat = np.vstack([
    df.loc[mask_zoom][mask_A.values].mean().values,
    df.loc[mask_zoom][mask_B.values].mean().values
])

cluster_names = ["a", "b"]
cluster_df = pd.DataFrame(cluster_mat, index=cluster_names, columns=genes)

# ---------------------------------------------------
# 3. Fate-level averages (ADD Mono + Neu)
# ---------------------------------------------------
state = adata_cospar.obs["state_info"].astype(str)
state_order = [
    "Monocyte",
    "Neutrophil",
    "Meg",
    "Erythroid",
    "Mast",
    "Baso"
]

fate_rows = []
valid_fates = []

for s in state_order:
    mask = (state == s)
    if mask.sum() > 0:
        fate_rows.append(df[mask.values].mean().values)
        valid_fates.append(s)

fate_df = pd.DataFrame(fate_rows, index=valid_fates, columns=genes)

rename_map = {
    "Monocyte": "Mono",
    "Neutrophil": "Neu",
    "Meg": "Meg",
    "Erythroid": "Ery",
    "Mast": "Mast",
    "Baso": "Baso"
}

fate_df.index = [rename_map[x] for x in fate_df.index]

# ---------------------------------------------------
# 4. COSPAR-style normalization (per gene → sum to 1)
# ---------------------------------------------------
def normalize_relative(mat):
    mat = mat.values
    mat = (mat + 1e-10) / (mat.sum(axis=0, keepdims=True) + 1e-10)
    return pd.DataFrame(mat, index=None, columns=None)

cluster_rel = normalize_relative(cluster_df)
cluster_rel.index = cluster_df.index
cluster_rel.columns = cluster_df.columns

fate_rel = normalize_relative(fate_df)
fate_rel.index = fate_df.index
fate_rel.columns = fate_df.columns

# ---------------------------------------------------
# 5. Sort genes by logFC
# ---------------------------------------------------
deg_sub = deg_df.set_index("gene").loc[genes]
gene_order = deg_sub["logFC"].sort_values(ascending=False).index.tolist()

cluster_rel = cluster_rel[gene_order]
fate_rel = fate_rel[gene_order]

# ---------------------------------------------------
# 6. TRANSPOSE
# ---------------------------------------------------
cluster_T = cluster_rel.T
fate_T = fate_rel.T

# ---------------------------------------------------
# 7. Plot side-by-side
# ---------------------------------------------------
fig = plt.figure(figsize=(4, 8))

gs = fig.add_gridspec(1, 2, width_ratios=[1, 3], wspace=0.03)

# --- Left: clusters ---
ax1 = fig.add_subplot(gs[0])

sns.heatmap(
    cluster_T,
    cmap=sns.diverging_palette(180, 20, as_cmap=True),
    vmin=0,
    vmax=1,
    center=0.5,
    cbar=False,
    ax=ax1
)

ax1.set_xlabel("")
ax1.set_ylabel("")
ax1.set_xticklabels(cluster_T.columns, rotation=45, fontsize=16)
ax1.set_yticklabels(cluster_T.index, fontsize=15)

# --- Right: fates ---
ax2 = fig.add_subplot(gs[1])

sns.heatmap(
    fate_T,
    cmap=sns.diverging_palette(180, 20, as_cmap=True),
    vmin=0,
    vmax=1,
    center=0.5,
    cbar=False,
    ax=ax2
)

ax2.set_xlabel("")
ax2.set_ylabel("")
ax2.set_xticklabels(fate_T.columns, fontsize=16)
plt.setp(
    ax2.get_xticklabels(),
    rotation=45,
    ha="right",
    rotation_mode="anchor"
)

# remove duplicate gene labels
ax2.set_yticks([])

plt.savefig(
    "./figures/larry/expression_heatmap.pdf",
    bbox_inches="tight",
    pad_inches=0.1
)
plt.show()

In [ ]:
# ---------------------------------------------------
# Thin standalone colorbar
# ---------------------------------------------------
import matplotlib as mpl

fig_cb, ax_cb = plt.subplots(figsize=(0.35, 4))

cmap = sns.diverging_palette(
    180,
    20,
    as_cmap=True
)

norm = mpl.colors.Normalize(
    vmin=0,
    vmax=1
)

cb = mpl.colorbar.ColorbarBase(
    ax_cb,
    cmap=cmap,
    norm=norm,
    orientation="vertical"
)

# ticks
cb.set_ticks([0, 0.5, 1.0])

cb.ax.tick_params(
    labelsize=16,
    length=2
)

# remove black outline
cb.outline.set_visible(False)

# optional: remove axis spines completely
for spine in ax_cb.spines.values():
    spine.set_visible(False)

cb.set_label(
    "Normalized nexpression",
    fontsize=16,
    labelpad=10
)

plt.savefig(
    "./figures/larry/expression_heatmap_colorbar.pdf",
    bbox_inches="tight",
    pad_inches=0.02,
    transparent=True,
)

plt.show()

In [ ]:
state
print(state.value_counts(), "\n")

In [ ]:
alpha = 0.05
logfc_cut = 1.0

# significance flags
df_clean = deg_df.dropna(subset=["logFC", "pval_adj"]).copy()
df_clean["-log10p"] = -np.log10(df_clean["pval_adj"].clip(lower=1e-300))

sig_both = (df_clean["pval_adj"] < alpha) & (df_clean["logFC"].abs() >= logfc_cut)

# pick top genes (annotation optional)
N_annotate = 0
df_sig_sorted = (
    df_clean.loc[sig_both]
             .sort_values("pval_adj")
             .head(N_annotate)
)

# ------------------------------------------------
# PLOT
# ------------------------------------------------
plt.figure(figsize=(6., 7.8))

# background
plt.scatter(
    df_clean.loc[~sig_both, "logFC"],
    df_clean.loc[~sig_both, "-log10p"],
    s=140, c="#C7C7C7", alpha=0.55, edgecolors="none"
)

# significant points
plt.scatter(
    df_clean.loc[sig_both, "logFC"],
    df_clean.loc[sig_both, "-log10p"],
    s=240, c="#B22222", alpha=0.9,
    edgecolors="black", linewidth=0.25
)

# cutoff lines
plt.axhline(-np.log10(alpha), color="black", linestyle="--", lw=1)
plt.axvline(logfc_cut, color="black", linestyle="--", lw=1)
plt.axvline(-logfc_cut, color="black", linestyle="--", lw=1)

# ------------------------------------------------
# OPTIONAL MANUAL ANNOTATION SECTION
# ------------------------------------------------
for _, row in df_sig_sorted.iterrows():
    plt.text(
        row["logFC"] + 0.10,
        row["-log10p"] + 0.10,
        row["gene"],
        fontsize=24,
        ha="left",
        va="bottom"
    )

# ------------------------------------------------
# Aesthetics
# ------------------------------------------------
ax = plt.gca()

# keep only axis lines
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
for spine in ["bottom", "left"]:
    ax.spines[spine].set_visible(True)
    ax.spines[spine].set_linewidth(1.3)

# axis labels
plt.xlabel(r"log$_2$ FC (a / b)", fontsize=36)
plt.ylabel(r"-log$_{10}$(adjusted p)", fontsize=36)

# ------------------------------------------------
# Ticks: increase size & sparsity
# ------------------------------------------------

# y ticks: 0, 30, 60
yticks = [0, 10, 20]
plt.yticks(yticks, [str(y) for y in yticks], fontsize=24)

# x ticks: spaced, larger
plt.xticks([-10, -5, 0, 5, 10], fontsize=24)

plt.xlim(-12, 12)
plt.grid(False)
plt.tick_params(axis="both", length=5, width=1.2, color="black")

plt.tight_layout()
plt.show()

In [ ]:
genes = [
    "Npm1", "Set", "C1qbp", "Hspd1",
    "Hspa9", "Ptprcap", "Cd34", "Nolc1"
]

adata_cospar.obs["cluster"] = np.nan
adata_cospar.obs.loc[mask_zoom, "cluster"] = labels_grow.astype(str)

# keep only clusters 0 and 1
clusters = adata_cospar.obs["cluster"].astype(str).values
mask = (clusters == "0") | (clusters == "1")
adata_sub = adata_cospar[mask]

# extract expression
X = adata_sub.X
if not isinstance(X, np.ndarray):
    X = X.A

clusters_sub = adata_sub.obs["cluster"].astype(str).values

# ---------------------------------------------------
# Build long-form dataframe for seaborn
# ---------------------------------------------------
df_plot = pd.concat([
    pd.DataFrame({
        "gene": g,
        "expression": X[:, np.where(adata_sub.var_names == g)[0][0]],
        "cluster": clusters_sub
    })
    for g in genes
], axis=0)

# ---------------------------------------------------
# Plot: 2 × 4 violin grid
# ---------------------------------------------------
fig, axes = plt.subplots(2, 4, figsize=(14, 6), sharey=False)
axes = axes.ravel()

for ax, g in zip(axes, genes):
    sns.violinplot(
        data=df_plot[df_plot["gene"] == g],
        x="cluster",
        y="expression",
        ax=ax,
        cut=0,
        inner="box",
        linewidth=1,
        palette=["#4e79a7", "#f28e2b"],
    )
    ax.set_title(g, fontsize=12)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------
# 1. Expression + labels from adata_cospar_zoom
# ------------------------------------------------------
X = adata_cospar_zoom.X
if not isinstance(X, np.ndarray):
    X = X.A

gene_names = np.array(adata_cospar_zoom.var_names)

clusters = adata_cospar_zoom.obs["cluster"].astype(str).values

# enforce exactly two clusters
clA, clB = "0", "1"

idx_A = np.where(clusters == clA)[0]
idx_B = np.where(clusters == clB)[0]

# ------------------------------------------------------
# 2. Top DE genes (already computed)
# ------------------------------------------------------
topN = 20
df_sub = deg_df.iloc[:topN].copy()

# Order genes by logFC:
#   <0 → enriched in A
#   >0 → enriched in B
# df_sub = df_sub.sort_values("logFC", ascending=True)

ordered_genes = df_sub["gene"].values
gene_idx = [np.where(gene_names == g)[0][0] for g in ordered_genes]

# ------------------------------------------------------
# 3. Extract expression matrices
# ------------------------------------------------------
expr_A = X[idx_A][:, gene_idx]
expr_B = X[idx_B][:, gene_idx]

expr_all = np.vstack([expr_A, expr_B]).astype(float)

# ------------------------------------------------------
# 4. Z-score per gene
# ------------------------------------------------------
mean = expr_all.mean(axis=0)
std = expr_all.std(axis=0) + 1e-8
expr_z = np.clip((expr_all - mean) / std, -2, 2)

# ------------------------------------------------------
# 5. Build heatmap matrix (A → B ordering)
# ------------------------------------------------------
cluster_track = (
    [clA] * expr_A.shape[0] +
    [clB] * expr_B.shape[0]
)

df = pd.DataFrame(expr_z, columns=ordered_genes)
df["cluster"] = cluster_track

df_heat = df[ordered_genes].T   # genes × cells

# ------------------------------------------------------
# 6. Minimal heatmap
# ------------------------------------------------------
palette = {clA: "#1f77b4", clB: "#d62728"}
col_colors = df["cluster"].map(palette).values

sns.set(style="white")

g = sns.clustermap(
    df_heat,
    row_cluster=True,
    col_cluster=False,
    col_colors=col_colors,
    cmap="viridis",
    vmin=-1, vmax=1,
    xticklabels=False,
    yticklabels=True,
    figsize=(12, 9),
)

plt.suptitle(
    f"Top {topN} DE Genes (Ordered by logFC, {clA} → {clB})",
    fontsize=16
)
plt.show()


In [ ]:
import cospar as cs

selected_fates = [
    ["Neutrophil", "Monocyte"],          # Group A (myeloid)
    ["Meg", "Erythroid", "Mast", "Baso"] # Group B (erythroid/mast/baso)
]

# -----------------------------------------------------------
# 1. Run fate map
# -----------------------------------------------------------
cs.tl.fate_map(
    adata_cospar,
    selected_fates=selected_fates,
    source="transition_map",
    map_backward=True
)

# -----------------------------------------------------------
# 2. Compute fate bias
# -----------------------------------------------------------
cs.tl.fate_bias(
    adata_cospar,
    selected_fates=selected_fates,
    source="transition_map",
    pseudo_count=0,
    sum_fate_prob_thresh=0.1,
)

In [ ]:
selected_fates = [
    ["Neutrophil", "Monocyte"],          # Group A (myeloid)
    ["Meg", "Erythroid", "Mast", "Baso"] # Group B (erythroid/mast/baso)
]

cs.tl.progenitor(
    adata_cospar,
    selected_fates=selected_fates,
    source="transition_map",
    map_backward=True,
    bias_threshold_A=0.5,
    bias_threshold_B=0.5,
    sum_fate_prob_thresh=0.2,
    avoid_target_states=True,
)

In [ ]:
adata_cospar

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

# ---------------------------------------------------
# 0. Configuration
# ---------------------------------------------------
genes = (
    deg_df
    .sort_values("pval_adj")
    .head(25)["gene"]
    .tolist()
)

genes = [g for g in genes if g in adata_cospar.var_names]

bias_key = "fate_bias_transition_map_Neutrophil_Monocyte*Meg_Erythroid_Mast_Baso"
prog_A = "progenitor_transition_map_Neutrophil_Monocyte"
prog_B = "progenitor_transition_map_Meg_Erythroid_Mast_Baso"

eps = 1e-6
n_grid = 300          # resolution of continuous axis
sigma = 30             # smoothing strength (increase = smoother)

# ---------------------------------------------------
# 1. Progenitor UNION mask
# ---------------------------------------------------
prog_mask = (
    adata_cospar.obs[prog_A].values.astype(bool)
    | adata_cospar.obs[prog_B].values.astype(bool)
)

# ---------------------------------------------------
# 2. Extract bias + expression
# ---------------------------------------------------
bias = adata_cospar.obs[bias_key].values[prog_mask]

valid = np.abs(bias - 0.5) > eps
bias = bias[valid]

X = adata_cospar[prog_mask, genes].X
if hasattr(X, "A"):
    X = X.A
X = X[valid]

# ---------------------------------------------------
# 3. Sort by fate bias
# ---------------------------------------------------
order = np.argsort(bias)
bias = bias[order]
X = X[order]

# ---------------------------------------------------
# 4. Smooth expression along fate bias
# ---------------------------------------------------
# regular bias grid
bias_grid = np.linspace(0, 1, n_grid)

X_smooth = np.zeros((len(genes), n_grid))

for gi in range(len(genes)):
    # interpolate gene expression onto grid
    X_interp = np.interp(bias_grid, bias, X[:, gi])
    
    # smooth along bias axis
    X_smooth[gi] = gaussian_filter1d(X_interp, sigma=sigma)

# ---------------------------------------------------
# 5. Z-score per gene (row-wise)
# ---------------------------------------------------
mean = X_smooth.mean(axis=1, keepdims=True)
std = X_smooth.std(axis=1, keepdims=True) + 1e-8
Xz = (X_smooth - mean) / std
Xz = np.clip(Xz, -2, 2)

from scipy.cluster.hierarchy import linkage, leaves_list

Z = linkage(Xz, method="average", metric="euclidean")
row_order = leaves_list(Z)

Xz_ord = Xz[row_order]
genes_ord = [genes[i] for i in row_order]


import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

fig, ax = plt.subplots(figsize=(5.5, 3.5))

sns.heatmap(
    Xz_ord,
    ax=ax,
    cmap="cividis",
    center=0,
    yticklabels=genes_ord,
    xticklabels=False,
    cbar=False,
    rasterized=True
)

# ---------------------------------------------------
# Fate-bias boundary
# ---------------------------------------------------
boundary_idx = np.argmin(np.abs(bias_grid - 0.5))
ax.axvline(boundary_idx, color="red", linewidth=4)

# ---------------------------------------------------
# X-axis ticks (fate bias)
# ---------------------------------------------------
target_ticks = np.array([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
tick_pos = [np.argmin(np.abs(bias_grid - t)) for t in target_ticks]

ax.set_xticks(tick_pos)
ax.set_xticklabels([f"{t:.1f}" for t in target_ticks], fontsize=12)

ax.set_xlabel("Fate bias (Mk/Er/Ma/Ba vs Neu/Mo)", fontsize=14)
ax.set_ylabel("")

ax.tick_params(axis="y", labelsize=9)

plt.tight_layout()

out_path = "./figures/larry/fate_bias_gene_heatmap.pdf"

fig.savefig(
    out_path,
    dpi=300,              # irrelevant for PDF, critical for PNG
    bbox_inches="tight",
    pad_inches=0.02
)

plt.show()


In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.cluster.hierarchy import linkage, leaves_list

# ---------------------------------------------------
# 0. Configuration
# ---------------------------------------------------
genes = (
    deg_df
    .sort_values("pval_adj")
    .head(25)["gene"]
    .tolist()
)

genes = [g for g in genes if g in adata_cospar.var_names]

prog_A = "progenitor_transition_map_Neutrophil_Monocyte"
prog_B = "progenitor_transition_map_Meg_Erythroid_Mast_Baso"

n_grid = 300
sigma = 3   # smaller now because binning already smooths

# ---------------------------------------------------
# 1. Progenitor UNION mask
# ---------------------------------------------------
prog_mask = (
    adata_cospar.obs[prog_A].values.astype(bool)
    | adata_cospar.obs[prog_B].values.astype(bool)
)

# ---------------------------------------------------
# 2. Extract fate probability + expression
# ---------------------------------------------------
pA = adata_cospar.obs["fate_map_transition_map_Neutrophil_Monocyte"].values
pB = adata_cospar.obs["fate_map_transition_map_Meg_Erythroid_Mast_Baso"].values

pA = pA[prog_mask]
pB = pB[prog_mask]

prob = pA / (pA + pB + 1e-8)

# filter low-confidence cells
valid = (pA + pB) > 1e-3
prob = prob[valid]

X = adata_cospar[prog_mask, genes].X
if hasattr(X, "A"):
    X = X.A
X = X[valid]

# ---------------------------------------------------
# 3. Sort by fate probability
# ---------------------------------------------------
order = np.argsort(prob)
prob = prob[order]
X = X[order]

# ---------------------------------------------------
# 4. Bin + smooth expression
# ---------------------------------------------------
prob_grid = np.linspace(0, 1, n_grid)

bin_idx = np.digitize(prob, prob_grid) - 1
bin_idx = np.clip(bin_idx, 0, n_grid - 1)

X_binned = np.zeros((len(genes), n_grid))
counts = np.zeros(n_grid)

for i in range(len(prob)):
    X_binned[:, bin_idx[i]] += X[i]
    counts[bin_idx[i]] += 1

counts[counts == 0] = 1
X_binned /= counts

# smooth along fate axis
X_smooth = gaussian_filter1d(X_binned, sigma=sigma, axis=1)

# ---------------------------------------------------
# 5. Z-score per gene
# ---------------------------------------------------
mean = X_smooth.mean(axis=1, keepdims=True)
std = X_smooth.std(axis=1, keepdims=True) + 1e-8
Xz = (X_smooth - mean) / std
Xz = np.clip(Xz, -2, 2)

# ---------------------------------------------------
# 6. Cluster genes
# ---------------------------------------------------
Z = linkage(Xz, method="average", metric="euclidean")
row_order = leaves_list(Z)

Xz_ord = Xz[row_order]
genes_ord = [genes[i] for i in row_order]

# ---------------------------------------------------
# 7. Plot
# ---------------------------------------------------
fig, ax = plt.subplots(figsize=(5.5, 3.5))

sns.heatmap(
    Xz_ord,
    ax=ax,
    cmap="RdBu_r",
    center=0,
    yticklabels=genes_ord,
    xticklabels=False,
    cbar=False,
    rasterized=True
)

# ---------------------------------------------------
# Boundary at probability = 0.5
# ---------------------------------------------------
boundary_idx = np.argmin(np.abs(prob_grid - 0.5))
ax.axvline(boundary_idx, color="black", linewidth=3)

# ---------------------------------------------------
# X-axis ticks
# ---------------------------------------------------
target_ticks = np.array([0.0, 0.25, 0.5, 0.75, 1.0])
tick_pos = [np.argmin(np.abs(prob_grid - t)) for t in target_ticks]

ax.set_xticks(tick_pos)
ax.set_xticklabels([f"{t:.2f}" for t in target_ticks], fontsize=11)

ax.set_xlabel("Fate probability (Neu/Mo → Mk/Er/Ma/Ba)", fontsize=13)
ax.set_ylabel("")
ax.tick_params(axis="y", labelsize=9)

# ---------------------------------------------------
# Clean layout
# ---------------------------------------------------
plt.tight_layout()

out_path = "./figures/larry/fate_probability_gene_heatmap.pdf"

fig.savefig(
    out_path,
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.02
)

plt.show()

In [ ]:
cs.tl.fate_map(
    adata_cospar,
    selected_fates=["Neutrophil", "Mast", "Meg", "Monocyte", "Baso",   "Erythroid"],
    source="transition_map",
    map_backward=True,
)

In [ ]:
from matplotlib.ticker import FuncFormatter
from scipy.stats import mannwhitneyu
import matplotlib.colors as mcolors

def lighten_color(color, amount=0.6):
    c = np.array(mcolors.to_rgb(color))
    return tuple(c + (1 - c) * amount)

def percent_fmt_sparse(x, _):
    p = x * 100
    if abs(p - 0.5) < 1e-6:
        return "0.5%"
    if abs(p - round(p)) < 1e-6:
        return f"{int(round(p))}%"
    return ""

def p_to_stars(p):
    if p < 1e-4:
        return "****"
    elif p < 1e-3:
        return "***"
    elif p < 1e-2:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"


# ---------------------------------------------------
# Configuration
# ---------------------------------------------------
cluster_map = {"0": "a", "1": "b"}
palette = {"a": "#4e79a7", "b": "#f28e2b"}

fate_grid = [
    ["Neutrophil", "Mast", "Meg"],
    ["Monocyte", "Baso",   "Erythroid"],
]

# Column-specific x-axis ranges
xlims = {
    0: (0.0, 0.04),   # Neutrophil / Monocyte
    1: (0.0, 0.01),   # Mast / Baso
    2: (0.0, 0.01),   # Meg / Erythroid
    
}

xticks = {
    0: [0.00, 0.02, 0.04],
    1: [0.00, 0.005, 0.01],
    2: [0.00, 0.005, 0.01],
}

percent_fmt = FuncFormatter(lambda x, _: f"{round(x * 100, 1)}%")

# ---------------------------------------------------
# 1. Subset + relabel clusters
# ---------------------------------------------------
clusters = adata_cospar.obs["cluster"].astype(str)
mask = clusters.isin(["0", "1"])
adata_sub = adata_cospar[mask]

# ---------------------------------------------------
# 2. Build long-form dataframe
# ---------------------------------------------------
df_plot = pd.concat([
    pd.DataFrame({
        "fate": fate,
        "probability": adata_sub.obs[f"fate_map_transition_map_{fate}"].values,
        "cluster": adata_sub.obs["cluster"].map(cluster_map).values,
    })
    for fate in np.ravel(fate_grid)
], axis=0)

# ---------------------------------------------------
# 3. Plot
# ---------------------------------------------------
# ---------------------------------------------------
# Plot
# ---------------------------------------------------
sns.set_theme(style="white", context="paper", font_scale=1.2)

fig, axes = plt.subplots(
    2, 3,
    figsize=(10, 6),
    sharey=True,
    sharex="col"
)

# lighter colors for violins
violin_palette = {
    k: lighten_color(v, amount=0.65)
    for k, v in palette.items()
}

for i in range(2):
    for j in range(3):
        fate = fate_grid[i][j]
        ax = axes[i, j]
        df_f = df_plot[df_plot["fate"] == fate]

        # ---------------- Violin (LIGHT) ----------------
        sns.violinplot(
            data=df_f,
            x="probability",
            y="cluster",
            ax=ax,
            cut=0,
            inner=None,
            linewidth=0,
            palette=violin_palette,
            orient="h",
            alpha=0.9,
            zorder=1,
        )

        # ---------------- Box (STRONG ORIGINAL COLOR) ----------------
        sns.boxplot(
            data=df_f,
            x="probability",
            y="cluster",
            ax=ax,
            width=0.15,
            showcaps=True,
            boxprops={
                "facecolor": "none",
                "linewidth": 3.5,
            },
            whiskerprops={
                "linewidth": 3.5,
            },
            capprops={
                "linewidth": 3.5,
            },
            medianprops={
                "color": "black",
                "linewidth": 5.0,
            },
            showfliers=False,
            orient="h",
            palette=palette,   # <- ORIGINAL COLORS HERE
            zorder=3,
        )

        # ---------------- Stats ----------------
        x_a = df_f.loc[df_f["cluster"] == "a", "probability"].dropna()
        x_b = df_f.loc[df_f["cluster"] == "b", "probability"].dropna()

        if len(x_a) >= 3 and len(x_b) >= 3:
            pval = mannwhitneyu(x_a, x_b, alternative="two-sided").pvalue
            stars = p_to_stars(pval)
        else:
            stars = "ns"

        # ax.text(
        #     xlims[j][1] * 0.95,
        #     0.5,
        #     stars,
        #     ha="right",
        #     va="center",
        #     fontsize=22,
        #     fontweight="bold",
        # )

        # ---------------- Axis formatting ----------------
        ax.set_xlim(*xlims[j])
        ax.set_xticks(xticks[j])
        ax.xaxis.set_major_formatter(FuncFormatter(percent_fmt_sparse))

        ax.set_title(fate, fontsize=30, pad=8)
        ax.tick_params(axis="x", labelsize=22)
        ax.tick_params(axis="y", labelsize=24)

        ax.set_xlabel("")
        ax.set_ylabel("")

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

# ---------------------------------------------------
# Final layout
# ---------------------------------------------------
plt.tight_layout()
plt.show()